# 16-01 CEM 搜索与滚动纠错

对应第 16 章。依赖 NumPy、Matplotlib，CPU 即可。本实验使用明确给出的二维位置更新规则，不训练视觉模型，不是 PushT。它隔离一个问题：真实状态被扰动以后，重新观察和规划有什么作用？

比较相同起点、目标、动作边界与一次相同扰动。两种方式都使用 CEM，但滚动方式重复搜索，计算总量更多。因此只能解释反馈机制，不能用这个实验宣称同等计算预算下的算法优越性。

## 1. 先把预测和评价写清楚

状态是二维位置，动作是每步位移，各轴限幅 0.12。预测器直接把动作依次加到当前位置，不包含接触、惯性或障碍物。代价同时考虑终点距离、沿途距离和动作幅度，越小越好。CEM 每轮采样 384 条、保留 38 条较好候选，重复 5 轮。

本例从目标方向初始化采样均值，以减少演示所需计算；不是从零先验的通用规划器。

In [ ]:
import numpy as np  # 计算批量候选轨迹
import matplotlib.pyplot as plt  # 显示路径与误差
rng = np.random.default_rng(16)  # 固定搜索随机数以便复查结果
goal = np.array([0.45, 0.3])  # 指定二维目标位置
start = np.array([-0.45, -0.3])  # 指定两种控制方式的共同起点
limit = 0.12  # 限制每个动作坐标的最大位移
horizon = 12  # 每次预测未来十二步
def predict_paths(state, actions):  # 定义固定且已知的教学预测器
    return state + np.cumsum(actions, axis=1)  # 沿时间累加每条候选动作
def costs(state, actions):  # 定义所有候选共用的评价规则
    paths = predict_paths(state, actions)  # 先计算候选导致的位置变化
    distances = ((paths - goal) ** 2).sum(axis=2)  # 计算各步到目标的平方距离
    return distances[:, -1] + 0.15 * distances.mean(axis=1) + 0.02 * (actions ** 2).sum(axis=(1, 2))  # 同时评价终点、过程与控制幅度
def cem(state, steps=horizon):  # 在固定世界预测器上搜索动作而不训练网络
    mean = np.tile(np.clip((goal - state) / steps, -limit, limit), (steps, 1))  # 将直接接近目标作为可解释的初始化
    std = np.full_like(mean, 0.07)  # 给各步动作保留搜索空间
    best = mean.copy()  # 保存一个已经评分过的具体候选
    best_score = costs(state, best[None])[0]  # 计算初始化候选的真实规划代价
    history = []  # 保存每轮搜索后的最低已知代价
    for iteration in range(5):  # 固定每次搜索的迭代预算
        samples = np.clip(rng.normal(mean, std, size=(384, steps, 2)), -limit, limit)  # 采样并保证动作合法
        samples[0] = best  # 保留上一轮最好的候选避免遗忘
        scores = costs(state, samples)  # 使用同一个模型与评分规则比较候选
        elite_ids = np.argsort(scores)[:38]  # 选出本轮代价最低的一小部分
        elite = samples[elite_ids]  # 取得具体动作序列而不是平均预测图像
        mean = elite.mean(axis=0)  # 更新每个动作位置的采样中心
        std = np.maximum(elite.std(axis=0), 0.008)  # 保留标准差下限避免搜索过早停止
        if scores[elite_ids[0]] < best_score:  # 仅当找到更好的具体动作时替换保留结果
            best = elite[0].copy()  # 保存实际评估过的动作块
            best_score = scores[elite_ids[0]]  # 保存对应的代价
        history.append(best_score)  # 记录截至当前的最低代价
    return best, history  # 返回动作与搜索过程而不是模型参数
plan, search_history = cem(start)  # 从共同起点搜索一次开环计划
fig, ax = plt.subplots(figsize=(6, 3))  # 创建搜索代价图
ax.plot(range(1, 6), search_history, marker="o")  # 显示每轮保留的最佳候选代价
ax.set(xlabel="CEM iteration", ylabel="Best cost so far")  # 说明横轴不是神经网络训练轮数
plt.show()  # 显示搜索逐步聚焦的结果


### 第一张图不是训练损失

横轴是当前决策内部的 CEM 迭代。预测规则始终不变，变化的是采样动作分布。由于显式保留历史最好候选，曲线不应上升；下降说明找到了代价更低的动作，水平段则说明本轮没有改善。

代价由多个项组成，不能把数值直接读成米，也不能把它当成真实成功率。

## 2. 两种方式遇到同一次扰动

在执行完第 6 步后，将位置额外移动 [-0.22, 0.20]。这次移动不在预测器内，也不提前告诉规划器。开环方式继续执行原计划；滚动方式每两步重新读取实际位置，再搜索未来十二步。扰动恰好发生在一个重规划边界，因此下次决策能看到它。

In [ ]:
disturbance = np.array([-0.22, 0.20])  # 定义两种方式共同经历的外部位移
open_states = [start.copy()]  # 保存开环执行轨迹
closed_states = [start.copy()]  # 保存滚动执行轨迹
buffer = None  # 保存滚动控制当前准备执行的两步动作
for step in range(12):  # 在相同长度内比较两种执行方式
    open_next = open_states[-1] + plan[step]  # 开环始终采用最初搜索到的动作
    if step % 2 == 0:  # 每执行两步便重新观察并规划
        buffer, _ = cem(closed_states[-1])  # 从最新真实位置重新搜索
    closed_next = closed_states[-1] + buffer[step % 2]  # 只执行新计划的前两步
    if step == 5:  # 在第六步结束时施加外部扰动
        open_next = open_next + disturbance  # 扰动开环系统的实际位置
        closed_next = closed_next + disturbance  # 对滚动系统施加同样的扰动
    open_states.append(open_next)  # 保存实际开环位置而不是模型想象
    closed_states.append(closed_next)  # 保存实际滚动控制位置
open_states = np.asarray(open_states)  # 整理开环轨迹
closed_states = np.asarray(closed_states)  # 整理闭环轨迹
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))  # 用路径与距离两幅图解释反馈
axes[0].plot(open_states[:, 0], open_states[:, 1], "o--", label="Open loop", markersize=3)  # 画出开环实际路径
axes[0].plot(closed_states[:, 0], closed_states[:, 1], "o-", label="Replan every 2 steps", markersize=3)  # 画出滚动实际路径
axes[0].scatter(*goal, marker="*", s=140, color="black", label="Goal")  # 标明目标位置
axes[0].set(xlabel="x", ylabel="y", aspect="equal")  # 保持横纵比例一致
axes[0].legend(fontsize=8)  # 显示路径含义
open_error = np.linalg.norm(open_states - goal, axis=1)  # 计算实际开环位置与目标的距离
closed_error = np.linalg.norm(closed_states - goal, axis=1)  # 计算实际滚动位置与目标的距离
axes[1].plot(open_error, "--", label="Open loop")  # 显示开环距离随时间变化
axes[1].plot(closed_error, label="Replanning")  # 显示滚动距离随时间变化
axes[1].axvline(6, color="gray", linestyle=":", label="Disturbance")  # 标出扰动发生的时刻
axes[1].set(xlabel="Step", ylabel="Distance to goal")  # 标明这里是物理距离而非规划总代价
axes[1].legend(fontsize=8)  # 标明曲线与事件
fig.tight_layout()  # 避免图例和标签挤在一起
plt.show()  # 展示同一扰动之后两种执行方式的分歧
print(f"最终距离：开环 {open_error[-1]:.3f}，滚动 {closed_error[-1]:.3f}")  # 报告实际计算出的最终误差
assert np.all(np.abs(plan) <= limit)  # 验证执行的开环计划满足本例动作范围
assert np.isfinite(closed_states).all()  # 验证滚动轨迹没有无效数值


### 最后两张图应连起来读

左图表示物体实际经过的位置，星号是目标。第 6 步的跳变来自外界扰动，不是 CEM 故意生成的大动作。右图的竖虚线对应同一事件。开环不知道起点已经改变，后续动作仍服务于旧计划；滚动方式读到新位置，随后把计划方向修正。

若滚动最终更接近目标，证明的是这个实验中的反馈确实有帮助，不是任意模型都安全。这里没有碰撞、没有模型学习误差，而且重规划多次消耗了更多计算。真实系统还须检查预测不准、计算超时与无效候选。

练习：将重新规划间隔从 2 改成 4，并同步修改取缓冲动作的余数；让扰动发生在两次更新之间，观察响应延迟。不要只改变条件而忘记缓冲索引。